In [ ]:
from dotenv import load_dotenv
import os
import requests

load_dotenv()  # Load variables from .env file

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError("GitHub token not found. Please set it in .env or your environment.")

BASE_URL = "https://api.github.com/graphql"

HEADERS = {
    'Authorization': f'bearer {GITHUB_TOKEN}',
    'Content-Type': 'application/json'
}

In [ ]:
# Define the GraphQL query
# States are MERGED and CLOSED cause in graphql 3 states: 1.merged (closed and merged), 2.closed (closed unmerged), 3.open
def make_query(owner, repo, cursor=None):
    return """
    {
      repository(owner: "%s", name: "%s") {
        pullRequests(states: [MERGED, CLOSED], first: 100 %s) {
          nodes {
            url
            merged
            closingIssuesReferences(first: 10) {
              nodes {
                url
              }
            }
          }
          pageInfo {
            endCursor
            hasNextPage
          }
        }
      }
       rateLimit {
    cost
      }
    }
    """ % (owner, repo, f', after: "{cursor}"' if cursor else '')

In [ ]:
# Send the created query
def run_query(query):
    request = requests.post(BASE_URL, json={'query': query}, headers=HEADERS)
    if request.status_code == 200:
        return request.json()
    else:
        raise Exception(f"Query failed to run by returning code of {request.status_code}. {query}")

In [ ]:
# Sends the query and writes the result to arrays
def get_closed_prs_with_issues(owner, repo):
    merged_pr_issue_pairs = []
    unmerged_pr_issue_pairs = []
    has_next_page = True
    end_cursor = None

    while has_next_page:
        result = run_query(make_query(owner, repo, end_cursor))
        prs = result['data']['repository']['pullRequests']['nodes']
        page_info = result['data']['repository']['pullRequests']['pageInfo']

        for pr in prs:
            if pr['closingIssuesReferences']['nodes']:
                for issue in pr['closingIssuesReferences']['nodes']:
                    if pr['merged']:
                        merged_pr_issue_pairs.append((pr['url'], issue['url']))
                    else:
                        unmerged_pr_issue_pairs.append((pr['url'], issue['url']))

        has_next_page = page_info['hasNextPage']
        end_cursor = page_info['endCursor']

    return merged_pr_issue_pairs, unmerged_pr_issue_pairs

In [ ]:
# Get repository information
REPO_OWNER = 'huggingface'
REPO_NAME = 'transformers'

# Get the pairs of closed PRs and linked issues
merged_pr_issue_pairs, unmerged_pr_issue_pairs = get_closed_prs_with_issues(REPO_OWNER, REPO_NAME)

# Display the length
print("Transformers Library")
print(f"Total number of closed merged PRs linked to issues: {len(merged_pr_issue_pairs)}")
print(f"Total number of closed unmerged PRs linked to issues: {len(unmerged_pr_issue_pairs)}")

In [ ]:
import random

# Get the indices of the merged and unmerged pairs for one project
merged_indices = random.sample(range(len(merged_pr_issue_pairs)), merged_in_sample)
unmerged_indices = random.sample(range(len(unmerged_pr_issue_pairs)), unmerged_in_sample)

# Create new variables with the randomly selected pairs
merged_pairs = [merged_pr_issue_pairs[i] for i in merged_indices]
unmerged_pairs = [unmerged_pr_issue_pairs[i] for i in unmerged_indices]

# Print the lengths of the new variables
print(f"Length of merged_pairs: {len(merged_pairs)}")
print(f"Length of unmerged_pairs: {len(unmerged_pairs)}")

In [ ]:
# Save indices in csv
import csv
from google.colab import files

def create_csv_with_pr_issue_data(file_path, merged_pairs, unmerged_pairs):
    with open(file_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)

        # Write header row with URLs
        writer.writerow(["PR URL", "Issue URL", "Merged/Unmerged"])

        # Write merged pairs with full URLs
        for pr_url, issue_url in merged_pairs:
            writer.writerow([pr_url, issue_url, "Merged", "", ""])

        # Write unmerged pairs with full URLs
        for pr_url, issue_url in unmerged_pairs:
            writer.writerow([pr_url, issue_url, "Unmerged", "", ""])

    print(f"CSV file created and saved as {file_path}")
    try:
      from google.colab import files
      files.download(file_path)
    except ImportError:
      print("Not in Google Colab. Skipping download.")


# Define the path for the CSV file
csv_file_path = "PR_issue_pairs.csv"

# Create the CSV file with full URLs
create_csv_with_pr_issue_data(csv_file_path, merged_pairs, unmerged_pairs)

In [ ]:
# Merge CSVs containing PR-Issue pair urls of different projects

from google.colab import drive
import pandas as pd
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set paths (update folder if needed)
drive_path = '/content/drive/MyDrive/'  # Change if CSVs are in a subfolder
input_files = [
    os.path.join(drive_path, 'PR_issue_pairs2.csv'),
    os.path.join(drive_path, 'PR_issue_pairs3.csv')
]
output_file = os.path.join(drive_path, 'COMBINED_PR_ISSUE_LINKS2.csv')

# 3. Custom CSV reader to protect URLs
def safe_read_csv(file):
    return pd.read_csv(
        file,
        usecols=['PR URL', 'Issue URL', 'Merged/Unmerged'],  # Enforce your columns
        dtype={'PR URL': 'string', 'Issue URL': 'string'},    # Force string type
        keep_default_na=False                                 # Prevent NaN conversion
    )

# 4. Merge files
combined_df = pd.concat([safe_read_csv(f) for f in input_files], ignore_index=True)

# 5. Final validation
print("Merge Successful")
print(f"Total Rows: {len(combined_df)}")
print("\nSample Data:")
display(combined_df.head(3))

# 6. Save back to Drive (with URL protection)
combined_df.to_csv(output_file, index=False, encoding='utf-8')

# 7. Optional download
from google.colab import files
files.download(output_file)  # Remove if not needed

In [ ]:
# Detailed artifact retrieval

import os
import json
import time
import random
import argparse
import requests
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv

# === Load .env variables if available ===
load_dotenv()

# === Configurable Defaults ===
DEFAULT_INPUT_CSV = os.getenv("INPUT_CSV", "./COMBINED_PR_ISSUE_LINKS2.csv")
DEFAULT_OUTPUT_CSV = os.getenv("OUTPUT_CSV", "./PR_ISSUE_DATASET.csv")
DEFAULT_LOG_FILE = os.getenv("LOG_FILE", "./github_metadata_fetch_logs.txt")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError("GitHub token not found. Set GITHUB_TOKEN in environment or .env file.")

HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

RATE_LIMIT_DELAY = float(os.getenv("RATE_LIMIT_DELAY", 1.5))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", 3))

# === Logging ===
def log_message(message, log_file):
    timestamp = datetime.now().isoformat()
    entry = f"{timestamp} - {message}"
    print(entry)
    with open(log_file, "a") as f:
        f.write(entry + "\n")

# === GitHub API Request with Retry and Rate Limit Handling ===
def make_request(url, log_file, retry_count=0):
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)

        if response.status_code == 403:
            reset_time = int(response.headers.get('X-RateLimit-Reset', time.time() + 60))
            sleep_time = max(reset_time - time.time(), 0) + 5
            log_message(f"Rate limited. Sleeping for {sleep_time:.1f} seconds...", log_file)
            time.sleep(sleep_time)
            return make_request(url, log_file)

        if response.status_code >= 400:
            if retry_count < MAX_RETRIES:
                wait_time = (2 ** retry_count) + random.random()
                log_message(f"Retry {retry_count + 1} for {url} after {wait_time:.2f}s", log_file)
                time.sleep(wait_time)
                return make_request(url, log_file, retry_count + 1)
            else:
                log_message(f"Failed after {MAX_RETRIES} retries: {url}", log_file)
                return None

        return response

    except Exception as e:
        log_message(f"Request failed for {url}: {str(e)}", log_file)
        return None

# === Contributor Stats ===
def get_contributor_stats(login, log_file):
    try:
        total_url = f"https://api.github.com/search/issues?q=author:{login}+type:pr"
        merged_url = f"https://api.github.com/search/issues?q=author:{login}+type:pr+is:merged"

        total_res = make_request(total_url, log_file)
        merged_res = make_request(merged_url, log_file)

        total = total_res.json().get('total_count', 0) if total_res else 0
        merged = merged_res.json().get('total_count', 0) if merged_res else 0

        rate = f"{(merged / total * 100):.2f}%" if total > 0 else "N/A"

        return {
            'contributor': login,
            'total_prs': total,
            'merged_prs': merged,
            'acceptance_rate': rate
        }

    except Exception as e:
        log_message(f"Error getting stats for {login}: {str(e)}", log_file)
        return {
            'contributor': login,
            'total_prs': 0,
            'merged_prs': 0,
            'acceptance_rate': "N/A"
        }

# === PR Metadata ===
def get_pr_metadata(pr_url, log_file):
    try:
        api_url = pr_url.replace("github.com", "api.github.com/repos").replace("pull", "pulls")
        response = make_request(api_url, log_file)
        if not response:
            return None

        pr = response.json()
        project = "/".join(pr_url.split("/")[3:5])

        contributor_stats = get_contributor_stats(pr['user']['login'], log_file)

        commits_url = pr['_links']['commits']['href']
        comments_url = pr['_links']['comments']['href']
        review_comments_url = pr['_links']['review_comments']['href']
        diff_url = pr['diff_url']

        commits = make_request(commits_url, log_file)
        comments = make_request(comments_url, log_file)
        review_comments = make_request(review_comments_url, log_file)
        diff = make_request(diff_url, log_file)

        return {
            'project': project,
            'pr_url': pr_url,
            'title': pr.get('title', ''),
            'body': pr.get('body', ''),
            'pr_commit_messages': json.dumps([c['commit']['message'] for c in commits.json()] if commits else []),
            'pr_comments': json.dumps([c['body'] for c in comments.json()] if comments else []),
            'pr_review_comments': json.dumps([c['body'] for c in review_comments.json()] if review_comments else []),
            'lines_changed': pr.get('additions', 0) + pr.get('deletions', 0),
            'pr_diff': diff.text if diff else '',
            'pr_merged': pr.get('merged', False),
            'contributor_stats': json.dumps(contributor_stats)
        }

    except Exception as e:
        log_message(f"Error processing PR {pr_url}: {str(e)}", log_file)
        return None

# === Issue Metadata ===
def get_issue_metadata(issue_url, log_file):
    try:
        api_url = issue_url.replace("github.com", "api.github.com/repos")
        response = make_request(api_url, log_file)
        if not response:
            return None

        issue = response.json()
        return {
            'issue_links': issue_url,
            'issue_titles': issue.get('title', ''),
            'issue_description': issue.get('body', '')
        }

    except Exception as e:
        log_message(f"Error processing issue {issue_url}: {str(e)}", log_file)
        return None

# === Main Function ===
def main(input_csv, output_csv, log_file):
    log_message("🚀 Starting metadata collection", log_file)

    try:
        df = pd.read_csv(input_csv)
        log_message(f"Loaded {len(df)} records from {input_csv}", log_file)
    except Exception as e:
        log_message(f"Failed to load input CSV: {str(e)}", log_file)
        return

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing PRs"):
        if pd.isna(row.get('PR URL')):
            continue

        pr_data = get_pr_metadata(row['PR URL'], log_file)
        time.sleep(RATE_LIMIT_DELAY)

        if not pr_data:
            continue

        if pd.notna(row.get('Issue URL')):
            issue_data = get_issue_metadata(row['Issue URL'], log_file)
            time.sleep(RATE_LIMIT_DELAY)
            if issue_data:
                pr_data.update(issue_data)

        results.append(pr_data)

    if results:
        fields = [
            'project', 'pr_url', 'title', 'body', 'pr_commit_messages',
            'pr_comments', 'pr_review_comments', 'lines_changed',
            'pr_diff', 'pr_merged', 'contributor_stats',
            'issue_links', 'issue_titles', 'issue_description'
        ]
        pd.DataFrame(results)[fields].to_csv(output_csv, index=False)
        log_message(f"Saved metadata for {len(results)} PRs to {output_csv}", log_file)
    else:
        log_message("No results to save", log_file)

# === CLI Entrypoint ===
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', help='Path to input CSV', default=DEFAULT_INPUT_CSV)
    parser.add_argument('--output', help='Path to output CSV', default=DEFAULT_OUTPUT_CSV)
    parser.add_argument('--log', help='Path to log file', default=DEFAULT_LOG_FILE)
    args = parser.parse_args()

    main(args.input, args.output, args.log)
